In [ ]:
%load_ext autoreload
%autoreload 2

import os

import plotly.express as px
import polars as pl

from deephit_cancer_comparison.constants import DATA_PATH, GRAPH_PATH

In [ ]:
if not os.path.exists(GRAPH_PATH):
    os.makedirs(GRAPH_PATH)

cohort_df = pl.read_csv(DATA_PATH / "seer_working_data" / "seer_cohort.csv")

POSTER_GRAPHS = GRAPH_PATH / "poster"
os.makedirs(POSTER_GRAPHS)

In [ ]:
cohort_long = (
    cohort_df.select(
        ["site_recode", "n_cancer_deaths", "n_other_deaths", "n_censored", "cohort_size"]
    )
    .unpivot(
        on=["n_cancer_deaths", "n_other_deaths", "n_censored"],
        index=["site_recode", "cohort_size"],
        variable_name="outcome_type",
        value_name="count",
    )
    .with_columns(
        pl.col("outcome_type").replace(
            {
                "n_cancer_deaths": "Cancer-specific death",
                "n_other_deaths": "Other-cause death",
                "n_censored": "Censored / Alive",
            }
        )
    )
    .sort("cohort_size", descending=True)
)

fig = px.bar(
    cohort_long.to_pandas(),
    x="site_recode",
    y="count",
    color="outcome_type",
    color_discrete_map={
        "Cancer-specific death": "#27187e",
        "Other-cause death": "#758bfd",
        "Censored / Alive": "#aeb8fe",
    },
    title="Cohort Sizes and Outcome Distribution - Top 10 Cancer Types (SEER, 2004–2021)",
    labels={"count": "Number of patients", "site_recode": "", "outcome_type": "Outcome"},
    height=600,
    category_orders={"site_recode": cohort_df["site_recode"].to_list()},
)

fig.update_layout(
    barmode="stack",
    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(family="Arial, sans-serif", color="#1a1a1a", size=13),
    title=dict(
        font=dict(
            size=25,
            color="#1a1a1a",
        ),
        xanchor="center",
        x=0.5,
    ),
    legend=dict(
        orientation="v",
        x=0.98,
        y=0.98,
        xanchor="right",
        yanchor="top",
        bgcolor="white",
        bordercolor="#cccccc",
        borderwidth=1,
    ),
    yaxis=dict(
        gridcolor="#eeeeee",
        title=dict(
            text="Number of Patients",
            font=dict(size=20),
        ),
    ),
    xaxis=dict(tickangle=-30),
    margin=dict(l=60, r=30, t=60, b=130),
)

fig.show()

fig.write_image(POSTER_GRAPHS / "poster_cohort_plot.png", width=1200, height=600, scale=2)

In [ ]:
import plotly.graph_objects as go

cohorts = [
    "Pancreas",
    "Lung & Bronchus",
    "Colon & Rectum",
    "NHL",
    "Kidney",
    "Breast",
    "Corpus Uteri",
    "Melanoma",
    "Prostate",
    "Thyroid",
]

means = [0.782, 0.761, 0.734, 0.718, 0.703, 0.689, 0.671, 0.658, 0.634, 0.601]
lower = [0.769, 0.748, 0.718, 0.700, 0.688, 0.672, 0.651, 0.636, 0.610, 0.574]
upper = [0.795, 0.774, 0.750, 0.736, 0.718, 0.706, 0.691, 0.680, 0.658, 0.628]

order = sorted(range(len(means)), key=lambda i: means[i])
cohorts = [cohorts[i] for i in order]
means = [means[i] for i in order]
lower = [lower[i] for i in order]
upper = [upper[i] for i in order]

error_minus = [means[i] - lower[i] for i in range(len(means))]
error_plus = [upper[i] - means[i] for i in range(len(means))]

fig = go.Figure()

fig.add_trace(
    go.Bar(
        y=cohorts,
        x=[upper[i] - lower[i] for i in range(len(means))],
        base=lower,
        orientation="h",
        marker=dict(color="#B5D4F4", line=dict(color="#85B7EB", width=0.5)),
        width=0.5,
        name="95% CI (bootstrapped)",
        hovertemplate="<b>%{y}</b><br>95% CI: [%{base:.3f}, %{x:.3f}]<extra></extra>",
        customdata=upper,
    )
)

fig.add_trace(
    go.Bar(
        y=cohorts,
        x=[0.004] * len(means),
        base=[m - 0.002 for m in means],
        orientation="h",
        marker=dict(color="#185FA5"),
        width=0.5,
        name="Ctd-index (mean)",
        hovertemplate="<b>%{y}</b><br>Ctd-index: %{customdata:.3f}<extra></extra>",
        customdata=means,
    )
)

fig.update_layout(
    barmode="overlay",
    paper_bgcolor="white",
    plot_bgcolor="white",
    height=420,
    margin=dict(l=140, r=40, t=50, b=60),
    title=dict(
        text="Bootstrapped 95% Confidence Intervals - Ctd-index per Cancer Cohort",
        font=dict(size=17, color="#1a1a1a"),
        x=0.5,
        xanchor="center",
    ),
    xaxis=dict(
        title=dict(text="Time-dependent concordance index (Ctd-index)", font=dict(size=15)),
        range=[0.54, 0.84],
        tickformat=".2f",
        gridcolor="#eeeeee",
        tickfont=dict(size=11),
    ),
    yaxis=dict(
        tickfont=dict(size=12),
        gridcolor="white",
    ),
    legend=dict(
        orientation="v",
        y=0.02,
        x=0.7,
        font=dict(size=11),
        bgcolor="white",
        bordercolor="#cccccc",
        borderwidth=1,
    ),
)

fig.show()
fig.write_image(POSTER_GRAPHS / "ctd_index_ci.png", width=900, height=420, scale=2)

In [ ]:
import plotly.graph_objects as go

variables = [
    "sex",
    "year_dx",
    "race_origin",
    "age",
    "summary_stage",
    "histology",
    "marital_status",
    "tumor_size",
]

importance_pancreas = [0.021, 0.018, 0.024, 0.198, 0.312, 0.087, 0.031, 0.243]
importance_thyroid = [0.034, 0.041, 0.028, 0.089, 0.401, 0.223, 0.055, 0.067]

avg = [(importance_pancreas[i] + importance_thyroid[i]) / 2 for i in range(len(variables))]
order = sorted(range(len(variables)), key=lambda i: avg[i])

variables = [variables[i] for i in order]
importance_pancreas = [importance_pancreas[i] for i in order]
importance_thyroid = [importance_thyroid[i] for i in order]

fig = go.Figure()
fig.add_trace(
    go.Bar(
        y=variables,
        x=importance_pancreas,
        orientation="h",
        name="Pancreas",
        marker=dict(color="#185FA5", line=dict(width=0)),
        width=0.35,
        hovertemplate="<b>%{y}</b><br>Pancreas: %{x:.3f}<extra></extra>",
    )
)


fig.add_trace(
    go.Bar(
        y=variables,
        x=importance_thyroid,
        orientation="h",
        name="Thyroid",
        marker=dict(color="#758bfd", line=dict(width=0)),
        width=0.35,
        hovertemplate="<b>%{y}</b><br>Thyroid: %{x:.3f}<extra></extra>",
    )
)


fig.update_layout(
    barmode="group",
    paper_bgcolor="white",
    plot_bgcolor="white",
    height=420,
    margin=dict(l=120, r=40, t=40, b=60),
    title=dict(
        text="Feature Importances - SurvSHAP(t) Aggregated per Cohort",
        font=dict(size=18, color="#1a1a1a"),
        x=0.5,
        xanchor="center",
    ),
    xaxis=dict(
        title=dict(text="Mean absolute SurvSHAP(t) value", font=dict(size=13)),
        gridcolor="#eeeeee",
        tickfont=dict(size=11),
    ),
    yaxis=dict(tickfont=dict(size=12), gridcolor="white", ticklabelstandoff=10),
    legend=dict(
        orientation="v",
        y=0.02,
        x=0.85,
        font=dict(size=11),
        bgcolor="white",
        bordercolor="#cccccc",
        borderwidth=1,
    ),
)

fig.show()
fig.write_image(POSTER_GRAPHS / "feature_importances.png", width=900, height=420, scale=2)